# DeepSeek工具调用简单教学示例

这个教程将一步一步展示如何使用DeepSeek API进行工具调用。

## 1. 导入必要的库

In [1]:
import json
import os
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI

# 加载环境变量
load_dotenv()

print("✅ 库导入完成")

✅ 库导入完成


## 2. 设置API配置

In [2]:
# 初始化OpenAI客户端，指向DeepSeek API
API_KEY = os.getenv("DEEPSEEK_API_KEY")

if not API_KEY:
    print("❌ 请在.env文件中设置 DEEPSEEK_API_KEY")
else:
    print(f"✅ API密钥已加载: {API_KEY[:8]}...{API_KEY[-4:]}")

# 创建OpenAI客户端，指向DeepSeek API
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com"
)

✅ API密钥已加载: sk-6ee76...269a


## 3. 定义工具函数

我们定义几个简单的工具函数：

In [3]:
# 工具1: 获取当前时间
def get_current_time():
    """获取当前时间"""
    now = datetime.now()
    return {
        "current_time": now.strftime("%Y-%m-%d %H:%M:%S"),
        "timestamp": now.timestamp(),
        "weekday": now.strftime("%A")
    }

# 工具2: 简单计算器
def calculate(expression):
    """安全的数学计算"""
    try:
        # 只允许数字和基本运算符
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return {"error": "表达式包含不允许的字符"}
        
        result = eval(expression)
        return {
            "expression": expression,
            "result": result
        }
    except Exception as e:
        return {"error": str(e)}

# 工具3: 生成随机数
def generate_random_number(min_val=1, max_val=100):
    """生成指定范围内的随机数"""
    import random
    number = random.randint(min_val, max_val)
    return {
        "random_number": number,
        "range": f"{min_val}-{max_val}"
    }

print("✅ 工具函数定义完成")

✅ 工具函数定义完成


## 4. 测试工具函数

先直接测试一下我们的工具函数：

In [4]:
print("🧪 测试工具函数:")
print()

# 测试时间工具
print("1. 当前时间:")
time_result = get_current_time()
print(json.dumps(time_result, ensure_ascii=False, indent=2))
print()

# 测试计算工具
print("2. 计算 2 + 3 * 4:")
calc_result = calculate("2 + 3 * 4")
print(json.dumps(calc_result, ensure_ascii=False, indent=2))
print()

# 测试随机数工具
print("3. 生成1-10的随机数:")
random_result = generate_random_number(1, 10)
print(json.dumps(random_result, ensure_ascii=False, indent=2))


🧪 测试工具函数:

1. 当前时间:
{
  "current_time": "2025-07-04 13:43:19",
  "timestamp": 1751607799.597539,
  "weekday": "Friday"
}

2. 计算 2 + 3 * 4:
{
  "expression": "2 + 3 * 4",
  "result": 14
}

3. 生成1-10的随机数:
{
  "random_number": 5,
  "range": "1-10"
}


## 5. 定义工具描述

为了让AI知道如何使用这些工具，我们需要用JSON Schema格式描述它们：

In [5]:
# 工具描述 - 告诉AI如何使用这些工具
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "获取当前的日期和时间",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "计算数学表达式",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "要计算的数学表达式，如 '2+3*4'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_random_number",
            "description": "生成指定范围内的随机数",
            "parameters": {
                "type": "object",
                "properties": {
                    "min_val": {
                        "type": "integer",
                        "description": "最小值",
                        "default": 1
                    },
                    "max_val": {
                        "type": "integer",
                        "description": "最大值",
                        "default": 100
                    }
                },
                "required": []
            }
        }
    }
]

print("✅ 工具描述定义完成")
print(f"📋 共定义了 {len(tools)} 个工具")

✅ 工具描述定义完成
📋 共定义了 3 个工具


## 6. 工具执行函数

创建一个函数来执行AI选择的工具：

In [6]:
def execute_tool(tool_name, arguments):
    """执行指定的工具"""
    print(f"🔧 执行工具: {tool_name}")
    print(f"📝 参数: {arguments}")
    
    if tool_name == "get_current_time":
        result = get_current_time()
    elif tool_name == "calculate":
        result = calculate(arguments.get("expression", ""))
    elif tool_name == "generate_random_number":
        min_val = arguments.get("min_val", 1)
        max_val = arguments.get("max_val", 100)
        result = generate_random_number(min_val, max_val)
    else:
        result = {"error": f"未知工具: {tool_name}"}
    
    print(f"📊 结果: {json.dumps(result, ensure_ascii=False)}")
    return result

print("✅ 工具执行函数定义完成")

✅ 工具执行函数定义完成


## 7. 调用DeepSeek API

现在我们可以向DeepSeek发送带有工具的请求：

In [7]:
def call_deepseek_with_tools(user_message):
    """调用DeepSeek API并处理工具调用"""
    
    print(f"💬 用户消息: {user_message}")
    print("🚀 正在调用DeepSeek API...")
    
    try:
        # 使用OpenAI客户端调用DeepSeek API
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "user", "content": user_message}
                {""1. 对错误结果强行反思 - 工具出错
                   2. 对相似工具进行区分}
            ],
            tools=tools,
            tool_choice="auto"  # 让AI自动决定是否使用工具
        )
        
        message = response.choices[0].message
        
        # 检查是否有工具调用
        if message.tool_calls:
            print("\n🔧 AI决定使用工具:")
            
            # 执行每个工具调用
            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments)
                
                print(f"\n--- 工具调用 ---")
                execute_tool(tool_name, arguments)
        else:
            print("\n💬 AI回复:")
            print(message.content or "无回复内容")
        
        return response
        
    except Exception as e:
        print(f"❌ 错误: {e}")
        return None

print("✅ API调用函数定义完成")

✅ API调用函数定义完成


## 8. 示例1: 询问当前时间

In [8]:
# 测试时间查询
call_deepseek_with_tools("现在几点了？")

💬 用户消息: 现在几点了？
🚀 正在调用DeepSeek API...

🔧 AI决定使用工具:

--- 工具调用 ---
🔧 执行工具: get_current_time
📝 参数: {}
📊 结果: {"current_time": "2025-07-04 13:43:24", "timestamp": 1751607804.640915, "weekday": "Friday"}


ChatCompletion(id='ca99de37-ff84-41b6-a284-59caa06d77dc', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_0_a6d77d65-d156-4427-b96f-2b757af684bd', function=Function(arguments='{}', name='get_current_time'), type='function', index=0)]))], created=1751607800, model='deepseek-chat', object='chat.completion', service_tier=None, system_fingerprint='fp_8802369eaa_prod0623_fp8_kvcache', usage=CompletionUsage(completion_tokens=15, prompt_tokens=291, total_tokens=306, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=291))

## 9. 示例2: 数学计算

In [9]:
# 测试数学计算
call_deepseek_with_tools("帮我计算 15 * 8 + 32")

💬 用户消息: 帮我计算 15 * 8 + 32
🚀 正在调用DeepSeek API...

🔧 AI决定使用工具:

--- 工具调用 ---
🔧 执行工具: calculate
📝 参数: {'expression': '15*8+32'}
📊 结果: {"expression": "15*8+32", "result": 152}


ChatCompletion(id='732d71d1-cadf-4d80-a0ad-006596519d84', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_0_08bb414c-0a6f-46ad-ac3c-26c93d7c5545', function=Function(arguments='{"expression":"15*8+32"}', name='calculate'), type='function', index=0)]))], created=1751607804, model='deepseek-chat', object='chat.completion', service_tier=None, system_fingerprint='fp_8802369eaa_prod0623_fp8_kvcache', usage=CompletionUsage(completion_tokens=23, prompt_tokens=297, total_tokens=320, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=256), prompt_cache_hit_tokens=256, prompt_cache_miss_tokens=41))

## 10. 示例3: 生成随机数

In [10]:
# 测试随机数生成
call_deepseek_with_tools("给我一个1到50之间的随机数")

💬 用户消息: 给我一个1到50之间的随机数
🚀 正在调用DeepSeek API...

🔧 AI决定使用工具:

--- 工具调用 ---
🔧 执行工具: generate_random_number
📝 参数: {'min_val': 1, 'max_val': 50}
📊 结果: {"random_number": 24, "range": "1-50"}


ChatCompletion(id='f32a2490-ac49-4e12-bbbb-a6c3da37f4d7', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_0_3cac4b5c-0fb9-40bd-83db-4b748b63bf2a', function=Function(arguments='{"min_val":1,"max_val":50}', name='generate_random_number'), type='function', index=0)]))], created=1751607810, model='deepseek-chat', object='chat.completion', service_tier=None, system_fingerprint='fp_8802369eaa_prod0623_fp8_kvcache', usage=CompletionUsage(completion_tokens=26, prompt_tokens=295, total_tokens=321, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=256), prompt_cache_hit_tokens=256, prompt_cache_miss_tokens=39))

## 11. 示例4: 组合使用多个工具

In [11]:
# 测试多工具组合
call_deepseek_with_tools("告诉我现在的时间，然后计算 100 / 4，最后给我一个1到20的随机数")

💬 用户消息: 告诉我现在的时间，然后计算 100 / 4，最后给我一个1到20的随机数
🚀 正在调用DeepSeek API...

🔧 AI决定使用工具:

--- 工具调用 ---
🔧 执行工具: get_current_time
📝 参数: {}
📊 结果: {"current_time": "2025-07-04 13:43:41", "timestamp": 1751607821.923682, "weekday": "Friday"}

--- 工具调用 ---
🔧 执行工具: calculate
📝 参数: {'expression': '100 / 4'}
📊 结果: {"expression": "100 / 4", "result": 25.0}

--- 工具调用 ---
🔧 执行工具: generate_random_number
📝 参数: {'min_val': 1, 'max_val': 20}
📊 结果: {"random_number": 8, "range": "1-20"}


ChatCompletion(id='27206796-be0c-48eb-8573-3291cd352cc1', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_0_c3cde48f-6a3d-43ff-8087-c13a5d02b457', function=Function(arguments='{}', name='get_current_time'), type='function', index=0), ChatCompletionMessageToolCall(id='call_1_9a240ca5-d85d-475e-b871-423b50ac9d79', function=Function(arguments='{"expression": "100 / 4"}', name='calculate'), type='function', index=1), ChatCompletionMessageToolCall(id='call_2_6b75f088-1996-47f6-ad45-7ae0c6b9cb48', function=Function(arguments='{"min_val": 1, "max_val": 20}', name='generate_random_number'), type='function', index=2)]))], created=1751607815, model='deepseek-chat', object='chat.completion', service_tier=None, system_fingerprint='fp_8802369eaa_prod0623_fp8_kvcache', usage=CompletionUsage(completion_tokens

## 12. 示例5: 不需要工具的普通对话

In [12]:
# 测试普通对话（不使用工具）
call_deepseek_with_tools("你好，请介绍一下自己")

💬 用户消息: 你好，请介绍一下自己
🚀 正在调用DeepSeek API...

💬 AI回复:
你好！我是一个智能助手，可以帮助你完成各种任务和解答问题。以下是我的一些功能：

1. **信息查询**：我可以回答各种知识性问题，包括科学、历史、技术、文化等。
2. **数学计算**：无论是简单的加减乘除还是复杂的数学表达式，我都能帮你计算。
3. **时间查询**：可以告诉你当前的日期和时间。
4. **随机数生成**：如果你需要随机数，我可以为你生成指定范围内的数字。
5. **日常助手**：比如提醒、翻译、写作建议等。

如果你有任何问题或需要帮助，随时告诉我！ 😊


ChatCompletion(id='c6efc97d-e972-470a-996c-ba8705867fb4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='你好！我是一个智能助手，可以帮助你完成各种任务和解答问题。以下是我的一些功能：\n\n1. **信息查询**：我可以回答各种知识性问题，包括科学、历史、技术、文化等。\n2. **数学计算**：无论是简单的加减乘除还是复杂的数学表达式，我都能帮你计算。\n3. **时间查询**：可以告诉你当前的日期和时间。\n4. **随机数生成**：如果你需要随机数，我可以为你生成指定范围内的数字。\n5. **日常助手**：比如提醒、翻译、写作建议等。\n\n如果你有任何问题或需要帮助，随时告诉我！ 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1751607822, model='deepseek-chat', object='chat.completion', service_tier=None, system_fingerprint='fp_8802369eaa_prod0623_fp8_kvcache', usage=CompletionUsage(completion_tokens=126, prompt_tokens=292, total_tokens=418, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=256), prompt_cache_hit_tokens=256, prompt_cache_miss_tokens=36))

## 13. 自定义测试

修改下面的消息来测试你自己的问题：

In [13]:
# 在这里输入你想测试的消息
custom_message = "计算 (10 + 5) * 3，然后告诉我现在的时间"

call_deepseek_with_tools(custom_message)

💬 用户消息: 计算 (10 + 5) * 3，然后告诉我现在的时间
🚀 正在调用DeepSeek API...

🔧 AI决定使用工具:

--- 工具调用 ---
🔧 执行工具: calculate
📝 参数: {'expression': '(10 + 5) * 3'}
📊 结果: {"expression": "(10 + 5) * 3", "result": 45}

--- 工具调用 ---
🔧 执行工具: get_current_time
📝 参数: {}
📊 结果: {"current_time": "2025-07-04 13:43:57", "timestamp": 1751607837.091106, "weekday": "Friday"}


ChatCompletion(id='9ff5be1e-a698-4697-8068-acb09b63dd9a', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_0_d4876fa6-baa7-4da8-a793-5f90b734bd47', function=Function(arguments='{"expression": "(10 + 5) * 3"}', name='calculate'), type='function', index=0), ChatCompletionMessageToolCall(id='call_1_7e3c19b8-d2ae-40f7-8d20-2fd6732cc5c0', function=Function(arguments='{}', name='get_current_time'), type='function', index=1)]))], created=1751607831, model='deepseek-chat', object='chat.completion', service_tier=None, system_fingerprint='fp_8802369eaa_prod0623_fp8_kvcache', usage=CompletionUsage(completion_tokens=39, prompt_tokens=302, total_tokens=341, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=256), prompt_cache_hit_tokens=256, prompt_cach

## 14. 查看工具定义

让我们看看我们定义的工具长什么样：

In [14]:
print("🛠️ 我们定义的工具:")
print()

for i, tool in enumerate(tools, 1):
    func = tool["function"]
    print(f"{i}. {func['name']}")
    print(f"   描述: {func['description']}")
    
    params = func["parameters"]["properties"]
    if params:
        print(f"   参数: {list(params.keys())}")
    else:
        print(f"   参数: 无")
    print()

🛠️ 我们定义的工具:

1. get_current_time
   描述: 获取当前的日期和时间
   参数: 无

2. calculate
   描述: 计算数学表达式
   参数: ['expression']

3. generate_random_number
   描述: 生成指定范围内的随机数
   参数: ['min_val', 'max_val']



## 15. 工具调用的工作原理

让我们看看完整的请求数据是什么样的：

In [15]:
# 展示使用OpenAI客户端的调用方式
print("📋 使用OpenAI客户端调用DeepSeek API的方式:")
print()
print("client.chat.completions.create(")
print("    model='deepseek-chat',")
print("    messages=[{'role': 'user', 'content': '现在几点了？'}],")
print("    tools=tools,")
print("    tool_choice='auto'")
print(")")
print()
print("✅ 比手动构建requests请求简单多了！")

📋 使用OpenAI客户端调用DeepSeek API的方式:

client.chat.completions.create(
    model='deepseek-chat',
    messages=[{'role': 'user', 'content': '现在几点了？'}],
    tools=tools,
    tool_choice='auto'
)

✅ 比手动构建requests请求简单多了！


## 总结

这个简单的教学示例展示了DeepSeek工具调用的基本流程：

### 🔧 核心步骤

1. **定义工具函数** - 编写实际执行任务的Python函数
2. **描述工具** - 用JSON Schema告诉AI如何使用这些工具
3. **发送请求** - 将用户消息和工具描述一起发送给API
4. **处理响应** - 检查AI是否选择使用工具，如果是就执行相应的函数

### 📝 关键概念

- **工具描述**: 必须准确描述函数的功能和参数
- **参数验证**: 工具函数需要处理各种输入情况
- **错误处理**: 要考虑工具执行失败的情况
- **安全性**: 特别是像计算器这样的工具，需要验证输入

### 🚀 扩展建议

- 添加更多实用工具（文件操作、网络请求等）
- 实现工具调用的日志记录
- 添加工具权限控制
- 支持异步工具调用

现在你可以基于这个简单的框架，添加自己的工具函数了！